# **Conclusiones**

## Resumen del proyecto

ECGAssistant es un sistema de apoyo al diagnóstico cardíaco que combina
tres tecnologías de inteligencia artificial para clasificar señales ECG
y generar explicaciones clínicas en español.

El proyecto ha seguido el pipeline completo de un proyecto de IA real:
definición del problema, recolección de datos, preprocesamiento, EDA,
entrenamiento, evaluación, optimización y documentación.

## Resultados obtenidos


### CNN - Clasificación de señales ECG

| Métrica | Valor |
|---|---|
| F1-score (test) | 0.9867 |
| Accuracy (test) | 99% |
| AUC-ROC | 0.9988 |
| Recall clase anormal | 0.99 |
| Arritmias detectadas | 4.236 / 4.299 (98.5%) |
| Falsas alarmas | 51 / 14.748 (0.3%) |

El modelo supera con creces el objetivo marcado de F1 ≥ 0.95 y demuestra
una capacidad discriminativa casi perfecta con un AUC de 0.9988.

### LLM - Generación de explicaciones clínicas

Gemma 3 4B fine-tuneado con QLoRA sobre 200 pares pregunta-respuesta
clínicos en español. El modelo genera explicaciones coherentes, ancladas
en documentación médica real gracias al sistema RAG, y siempre incluye
el aviso de que no reemplaza la evaluación de un profesional médico.

### RAG - Recuperación de contexto clínico

Sistema de recuperación semántica con FAISS e índice de embeddings
multilingües. Permite que el LLM base sus respuestas en documentación
clínica verificada en lugar de generar información de memoria.

## Decisiones técnicas clave


**¿Por qué CNN 1D y no otros modelos?**
Las CNN 1D son especialmente adecuadas para series temporales como el ECG
porque aprenden directamente los patrones morfológicos de la señal
(complejo QRS, onda P, intervalo ST) sin necesidad de feature engineering
manual. Modelos como SVM o Random Forest requerirían extraer características
a mano, lo que introduciría sesgo humano en el proceso.

**¿Por qué Gemma 3 4B con QLoRA y no un modelo más grande?**
El objetivo era que el sistema fuera ejecutable en Colab gratuito.
QLoRA permite fine-tunear un modelo de 4B parámetros reduciendo el consumo
de VRAM de 16GB a menos de 5GB. Un modelo más grande mejoraría la calidad
de las explicaciones pero haría el sistema inviable sin infraestructura
de pago.

**¿Por qué RAG en lugar de incluir todo el contexto en el prompt?**
Con RAG el sistema escala independientemente del tamaño del corpus clínico.
Añadir nuevos documentos solo requiere regenerar el índice FAISS, sin
necesidad de reentrenar el modelo. Además, pasar documentos enteros al
LLM en cada consulta sería ineficiente y costoso en tokens.

**¿Por qué F1 y no accuracy como métrica principal?**
El dataset tiene un desbalance de 70/30. Un modelo que predijera siempre
normal obtendría un 70% de accuracy sin aprender nada. El F1 penaliza
los falsos negativos (arritmias no detectadas), que en contexto médico
son el error más crítico.

## Limitaciones del sistema

- **Clasificación binaria:** el sistema detecta si hay anomalía pero no
  identifica el tipo específico de arritmia. La granularidad diagnóstica
  la aporta el LLM a través del contexto RAG.
- **Una sola derivación:** entrenado sobre la derivación MLII del MIT-BIH.
  El rendimiento puede variar en ECGs de 12 derivaciones o dispositivos
  wearables.
- **Dataset histórico:** MIT-BIH fue recogido entre 1975 y 1979. Las señales
  modernas de dispositivos digitales pueden tener características diferentes.
- **Latidos individuales:** el sistema procesa latidos aislados de 288 muestras,
  no trazados completos. No detecta patrones que requieran analizar varios
  latidos consecutivos.
- **No es un dispositivo médico:** el sistema está diseñado como herramienta
  de apoyo, no como sustituto del criterio clínico de un profesional médico.

## Trabajo futuro

- **Clasificación multiclase:** extender el modelo para identificar los
  tipos específicos de arritmia además de la clasificación binaria.
- **Más derivaciones:** entrenar con ECGs de 12 derivaciones para mejorar
  la aplicabilidad clínica real.
- **Evaluación clínica formal:** validar las explicaciones del LLM con
  cardiólogos mediante rúbricas clínicas estructuradas.
- **Despliegue:** empaquetar el pipeline en una API REST que permita
  integrar ECGAssistant en sistemas de historia clínica electrónica.
- **Dataset más reciente:** reentrenar con datasets más modernos como
  PTB-XL (21.799 ECGs de 12 derivaciones) para mejorar la generalización.

## Reflexión del equipo

**¿Qué nos ha aportado desarrollar este proyecto?**
Hemos puesto en práctica de forma integrada los tres grandes bloques del
bootcamp: deep learning clásico con la CNN, LLMs con el fine-tuning de
Gemma, y técnicas de recuperación de información con RAG. Trabajar en un
dominio real como la cardiología nos ha obligado a entender el problema
a fondo antes de escribir código.

**¿Qué hemos aprendido?**
Que la parte técnica es solo una parte del trabajo. El preprocesamiento
correcto, entender el desbalance de clases, elegir la métrica adecuada
y documentar bien las decisiones son igual de importantes que la
arquitectura del modelo.

A nivel de equipo hemos aprendido a coordinarnos de forma real: dividir
el trabajo en historias de usuario, gestionar el backlog en Trello,
hacer seguimiento del progreso con el burndown chart y resolver conflictos
de código en GitHub. Son habilidades que en un proyecto individual nunca
hubiéramos practicado y que en el mundo laboral son tan importantes como
saber programar.

**¿Qué no volveríamos a hacer de la misma manera?**
Empezar con un notebook monolítico fue un error que nos costó tiempo
refactorizar. En un próximo proyecto definiríamos la estructura de
notebooks desde el principio.

También gestionaríamos mejor los tiempos en Trello desde el inicio,
siendo más realistas con las estimaciones de cada historia de usuario
y dejando margen para los imprevistos técnicos que siempre aparecen.

**¿Qué seguiremos haciendo en el futuro para mejorar el proyecto?**
Seguiremos usando GitHub con commits atómicos y mensajes descriptivos,
Trello para organizar el trabajo en sprints y la metodología SCRUM para
coordinarnos en equipo. Son herramientas y hábitos que nos han demostrado
su valor en este proyecto y que queremos consolidar en los siguientes.

A nivel técnico, añadir la evaluación clínica formal del LLM con rúbricas
validadas por médicos y explorar el despliegue del pipeline como API para
que sea usable en un entorno real.